In [1]:
from utilities import init_bigquery_client
from google.cloud import bigquery
import os
import pandas as pd
import numpy as np

from plotnine import *

#init BigQuery client
bq = init_bigquery_client()

Using BigQuery credentials: etl-testing-478716-c0b6c2c512e0.json


In [2]:
# Read from the 'events' table in BigQuery
query = """
    SELECT * FROM `etl-testing-478716.posthog_etl.events` 
    WHERE TIMESTAMP_TRUNC(timestamp, DAY) BETWEEN TIMESTAMP("2026-01-01") AND TIMESTAMP("2026-02-17")
"""
events_df = bq.query(query).to_dataframe()

query = """
    SELECT * FROM `etl-testing-478716.posthog_etl.sessions`
"""
sessions_df = bq.query(query).to_dataframe()
query = """
    SELECT *
    FROM `etl-testing-478716.firebase_etl_prod.users`
"""
users_df = bq.query(query).to_dataframe()

/opt/miniconda3/envs/heyyall/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.


In [3]:
import pandas as pd
import json

bok_ids = ['+18323900558', '+18323875995', '+18323787163', '+11111111111']
maaz_ids = []
taras_ids = []
zach_ids = ['+15126437937', '+15125577162']
trask_ids = ['+12146865810']
brandon_ids = ['+15126530534']

exclude_ids = bok_ids + maaz_ids + taras_ids + zach_ids + trask_ids + brandon_ids

# Parse properties JSON into new columns
def extract_properties(row):
    prop = json.loads(row['properties'])
    return pd.Series({
        'session_id': prop.get('$session_id'),
        'lib': prop.get('$lib'),
        'screen_name': prop.get('$screen_name'),
        'distinct_id': row['distinct_id'],  # assuming distinct_id is a flat column
    })

# Create new DataFrame with extracted columns
events_extracted = events_df.apply(extract_properties, axis=1)

# Filter to only "posthog-react-native" sessions and not excluded users
posthog_sessions = events_extracted[
    (events_extracted['lib'] == 'posthog-react-native') &
    (~events_extracted['distinct_id'].isin(exclude_ids)) &
    (events_extracted['session_id'].notnull())
]


# 1. Find session_ids where "Discover" was accessed
discover_session_ids = posthog_sessions[
    posthog_sessions['screen_name'] == 'Discover'
]['session_id'].unique()

# 2. Find session_ids where "Discover" was NOT accessed
all_session_ids = posthog_sessions['session_id'].unique()
non_discover_session_ids = list(set(all_session_ids) - set(discover_session_ids))

# 3. Filter sessions_df to not excluded users
sessions_filtered = sessions_df[
    (~sessions_df['distinct_id'].isin(exclude_ids))
]

# 4. Create Discover and Non-discover groups
discover_sessions = sessions_filtered[
    sessions_filtered['session_id'].isin(discover_session_ids)
]
non_discover_sessions = sessions_filtered[
    sessions_filtered['session_id'].isin(non_discover_session_ids)
]

# 5. Helper to aggregate
def aggregate_sessions(df):
    durations = df['session_duration']
    return {
        'sample_mean': durations.mean(),
        'sample_std_dev': durations.std(ddof=1),
        'sample_median': durations.median(),
        'sample_count': len(durations),
        'distinct_users': df['distinct_id'].nunique()
    }

discover_stats = aggregate_sessions(discover_sessions)
non_discover_stats = aggregate_sessions(non_discover_sessions)

# 6. Combine results
result = pd.DataFrame([
    {'session_type': 'Discover', **discover_stats},
    {'session_type': 'Non-discover', **non_discover_stats},
])

print(result)

   session_type  sample_mean  sample_std_dev  sample_median  sample_count  \
0      Discover   589.200000      220.861269          578.0             5   
1  Non-discover   241.850389      825.438751           19.0          1671   

   distinct_users  
0               5  
1             210  


In [4]:
discover_sessions = discover_sessions.merge(users_df, left_on='distinct_id', right_on='phoneNumber', how='left')

In [5]:
#convert start and end timetstamp to CST time
discover_sessions['start_timestamp_cst'] = discover_sessions['start_timestamp'] - pd.Timedelta(hours=6)
discover_sessions['end_timestamp_cst'] = discover_sessions['end_timestamp'] - pd.Timedelta(hours=6)

In [6]:
query = """
    SELECT *
    FROM `etl-testing-478716.firebase_etl_prod.quiz_answers`
"""
quiz_answer_df = bq.query(query).to_dataframe()
quiz_answer_df = quiz_answer_df.groupby('user_id').size().reset_index(name='quiz_answers_count').sort_values('quiz_answers_count', ascending=False)

/opt/miniconda3/envs/heyyall/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.


In [80]:
#filtering to ios sessions only
ios_sessions = sessions_filtered[sessions_filtered['session_id'].isin(posthog_sessions['session_id'])]

#iqr filtering ios sessions by session duration
Q1 = ios_sessions['session_duration'].quantile(0.25)
Q3 = ios_sessions['session_duration'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
ios_sessions_filtered = ios_sessions[(ios_sessions['session_duration'] >= lower_bound) & (ios_sessions['session_duration'] <= upper_bound)]

print(f"IQR filtering removed {len(ios_sessions) - len(ios_sessions_filtered)} sessions from the dataset.")

# Ensure start_timestamp is datetime
ios_sessions_filtered['start_timestamp'] = pd.to_datetime(ios_sessions_filtered['start_timestamp'])

# Create a new column for 5-minute intervals
ios_sessions_filtered['interval'] = ios_sessions_filtered['start_timestamp'].dt.floor('5min')

# Group by the interval and keep the last row in each group
grouped_ios_sessions = ios_sessions_filtered.groupby('interval').last().reset_index()

used_more_than_twice = grouped_ios_sessions.groupby('distinct_id').filter(lambda x: len(x) >= 2)['distinct_id'].unique()
used_more_than_120_seconds = grouped_ios_sessions[grouped_ios_sessions['session_duration'] > 120]['distinct_id'].unique()

# Find intersection of both sets
combo_users = set(used_more_than_twice) | set(used_more_than_120_seconds)

twice_dict = {}
for i in used_more_than_twice:
    twice_dict[i] = 'used_more_than_twice'
for i in used_more_than_120_seconds:
    if i in twice_dict:
        twice_dict[i] = 'used_more_than_twice_and_120_seconds'
    else:
        twice_dict[i] = 'used_more_than_120_seconds'

# Create a DataFrame from twice_dict
twice_df = pd.DataFrame(list(twice_dict.items()), columns=['phoneNumber', 'usage_status'])

# Merge with users_df on phoneNumber
users_with_usage = users_df.merge(twice_df, on='phoneNumber', how='inner').dropna(subset=['usage_status'])

print(f"Users with usage: {len(users_with_usage)}")

IQR filtering removed 276 sessions from the dataset.
Users with usage: 61


/var/folders/d6/wbxdxj_s71ngq7ft6m6_6krc0000gn/T/ipykernel_57437/1133864948.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/d6/wbxdxj_s71ngq7ft6m6_6krc0000gn/T/ipykernel_57437/1133864948.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [81]:
from plotnine import *

In [82]:
#output distribution of ios sessions by number of sessions and session duration
ios_sessions_filtered['session_count'] = ios_sessions_filtered.groupby('distinct_id')['session_id'].transform('count')
ios_sessions_filtered['session_duration_avg'] = ios_sessions_filtered.groupby('distinct_id')['session_duration'].transform('mean').round(2)

/var/folders/d6/wbxdxj_s71ngq7ft6m6_6krc0000gn/T/ipykernel_57437/1111746085.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/d6/wbxdxj_s71ngq7ft6m6_6krc0000gn/T/ipykernel_57437/1111746085.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [83]:
ios_sessions_filtered['session_count'].describe()

count    1400.000000
mean      152.001429
std       151.958334
min         1.000000
25%         9.000000
50%       123.000000
75%       374.000000
max       374.000000
Name: session_count, dtype: float64

In [84]:
ios_sessions_filtered['session_duration'].describe()

count       1400.0
mean     38.780714
std       60.03843
min            0.0
25%            4.0
50%           11.0
75%           42.0
max          287.0
Name: session_duration, dtype: Float64

In [85]:
high_usage = users_with_usage[users_with_usage['usage_status'].isin(['used_more_than_twice_and_120_seconds', 'used_more_than_twice'])]

In [86]:
per_user_stats = ios_sessions_filtered.drop_duplicates(subset=['distinct_id'])[['distinct_id', 'session_count', 'session_duration_avg']]

In [87]:
high_usage_stats = high_usage.merge(per_user_stats, left_on='phoneNumber', right_on='distinct_id', how='left')

In [77]:
# Read from the 'events' table in BigQuery
query = """
    SELECT *
    FROM `etl-testing-478716.firebase_etl_prod.events`
"""
events_df = bq.query(query).to_dataframe()

# Read from the 'userinvites' table in BigQuery
query = """
    SELECT *
    FROM `etl-testing-478716.firebase_etl_prod.userinvites`
"""
userinvites_df = bq.query(query).to_dataframe()

/opt/miniconda3/envs/heyyall/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.


In [78]:
high_usage_stats = high_usage_stats.merge(userinvites_df[userinvites_df['status'] == 'accepted'][['user_id', 'event_id']], left_on='user_id', right_on='user_id', how='left')

In [79]:
high_usage_stats = high_usage_stats.merge(events_df[['document_id', 'title']], left_on='event_id', right_on='document_id', how='left')